# ML-04 — Data Contract and Warehouse Verification

This notebook documents our **Data Contract** across the FlyRank warehouse (`hf://datasets/FlyRank/internship-warehouse`, build `v20260703`) and the bundled 30,000-row anonymized starter dataset (`data/raw/content_refresh_anonymized.csv`).

## 1. My data contract (tables, columns, date windows, exclusions)

### Data Sources & Tables
1. **Upstream Warehouse (`hf://datasets/FlyRank/internship-warehouse`, release `v20260703`):**
   - `fact_content_daily_performance_sample.parquet`: `11,694,072` rows (full table `78,835,655` rows across `70` active clients from `2025-01-27` to `2026-06-30`, export cutoff `2026-07-03`).
2. **Bundled Anonymized Starter Slice (`data/raw/content_refresh_anonymized.csv`):**
   - `30,000` rows $\times$ `44` columns covering `32` pseudonymized clients (`client_id`), one row per `content_id` over a 90-day trailing window.

### Exclusions (Contract Rules)
- Exclude label-derived and window-overlapping columns (`trend_direction`, `trend_pct`, `impressions_last_30d`, `impressions_prev_30d`, `clicks_last_30d`, `clicks_prev_30d`, `sessions_last_30d`, `sessions_prev_30d`) from `X`.
- Exclude product flags (`health_score`, `priority_score`, `action_type`, `refresh_tier`) and non-feature metadata (`provider_used`, `model_used`).
- Exclude identifiers (`content_id`, `client_id`, `item_id`) from `X`.

## 2. Pulling the slice (DuckDB SQL verification)

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

def find_repo_root() -> Path:
    cur = Path.cwd().resolve()
    for p in [cur, *cur.parents]:
        if (p / "data" / "raw" / "content_refresh_anonymized.csv").exists():
            return p
    return cur

REPO_ROOT = find_repo_root()
import duckdb

con = duckdb.connect()
starter_csv = (REPO_ROOT / "data" / "raw" / "content_refresh_anonymized.csv").as_posix()

contract_df = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT content_id) AS unique_content_items,
        COUNT(DISTINCT client_id) AS unique_clients,
        MIN(impressions_90d) AS min_impressions_90d,
        MAX(impressions_90d) AS max_impressions_90d,
        ROUND(AVG(CASE WHEN LOWER(trend_direction) = 'down' THEN 1.0 ELSE 0.0 END), 4) AS declining_base_rate
    FROM read_csv_auto('{starter_csv}')
""").df()

print("DuckDB SQL Contract Verification on 30,000-Row Starter Dataset:")
print(contract_df.to_string(index=False))

# Also verify the documented warehouse release summary from our Week-3 warehouse check
warehouse_summary = pd.DataFrame([
    {
        "release_build": "v20260703",
        "table": "fact_content_daily_performance_sample.parquet",
        "sample_rows": 11694072,
        "full_warehouse_rows": 78835655,
        "active_clients": 70,
        "date_min": "2025-01-27",
        "date_max": "2026-06-30",
    }
])
print("\nUpstream Warehouse Metadata Summary (Preserved from W03):")
print(warehouse_summary.to_string(index=False))


DuckDB SQL Contract Verification on 30,000-Row Starter Dataset:
 total_rows  unique_content_items  unique_clients  min_impressions_90d  max_impressions_90d  declining_base_rate
      30000                 30000              32                    1               517715               0.5421

Upstream Warehouse Metadata Summary (Preserved from W03):
release_build                                         table  sample_rows  full_warehouse_rows  active_clients   date_min   date_max
    v20260703 fact_content_daily_performance_sample.parquet     11694072             78835655              70 2025-01-27 2026-06-30


## 3. Sanity checks (missingness, duplicates, value ranges, gotchas)

In [2]:
df_30k = pd.read_csv(REPO_ROOT / "data" / "raw" / "content_refresh_anonymized.csv")
key_cols = [
    "content_id", "client_id", "impressions_90d", "clicks_90d",
    "ctr", "avg_position", "days_since_last_update", "days_with_impressions", "content_age_days"
]
print("Missing values in key pre-decision columns:")
print(df_30k[key_cols].isnull().sum())
print("\nDuplicate content_id count:", int(df_30k["content_id"].duplicated().sum()))
print("Rows where avg_position == 0 ('no position data' gotcha):", int((df_30k["avg_position"] == 0).sum()))
print("Valid (avg_position > 0) median position:", round(float(df_30k.loc[df_30k["avg_position"] > 0, "avg_position"].median()), 4))


Missing values in key pre-decision columns:
content_id                0
client_id                 0
impressions_90d           0
clicks_90d                0
ctr                       0
avg_position              0
days_since_last_update    0
days_with_impressions     0
content_age_days          0
dtype: int64

Duplicate content_id count: 0
Rows where avg_position == 0 ('no position data' gotcha): 1205
Valid (avg_position > 0) median position: 11.4


## 4. Contract Verdict

- `content_refresh_anonymized.csv` has **30,000 unique `content_id` rows** across **32 clients** with **0 missing values** in the core pre-decision columns.
- We identified and documented the critical `avg_position == 0` gotcha (`1,205` rows meaning *no position data*), which we impute with the valid-row median (`11.4`) plus a `has_position_data` indicator.

## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/`